In [1]:
import requests
import pandas as pd
import io

xml_url = "https://tfl.gov.uk/tfl/syndication/feeds/cycle-hire/livecyclehireupdates.xml"
resp = requests.get(xml_url, timeout=15)
resp.raise_for_status()

# attempt direct read; fallback to manual parsing if needed
try:
    df = pd.read_xml(io.BytesIO(resp.content), parser="lxml")
except Exception:
    import xml.etree.ElementTree as ET
    root = ET.fromstring(resp.content)
    # find the most common child tag (repeating records)
    tag_counts = {}
    for child in root:
        tag_counts[child.tag] = tag_counts.get(child.tag, 0) + 1
    rep_tag = max(tag_counts, key=tag_counts.get)
    rows = []
    for elem in root.findall(rep_tag):
        rows.append({c.tag: c.text for c in elem})
    df = pd.DataFrame(rows)

# Removal of apostrophe "'" and dot "." from station names. 
cleaned_names = df['name'].astype(str).str.replace("'", "", regex=False).str.replace(".", "", regex=False).str.strip()
# Next, split text by ",". store the text before "," in current column and store the text after "," in a new column called "District" 
df['name'] = cleaned_names.str.split(',', n=1).str[0].str.strip()
df.insert(2, 'district', cleaned_names.str.split(',', n=1).str[1].str.strip())

# fill na installation dates with 2015-01-01 00:00:00
df['installDate'] = df['installDate'].fillna('1420070400000')

df['installDate'] = pd.to_datetime(df['installDate'].astype(int), unit='ms').dt.strftime('%Y-%m-%d')
df['removalDate'] = pd.to_datetime(pd.to_numeric(df['removalDate'], errors='coerce'), unit='ms').dt.strftime('%Y-%m-%d')
print(df.shape)
df.head(10)
# output to a csv file on local drive
# Example Windows absolute path (using a raw string)
# windows_path = r'C:\Users\YourUser\Documents\sales_data.csv'
# df.to_csv(windows_path, index=False)
# Example macOS/Linux absolute path
# mac_linux_path = '/Users/YourUserid/Documents/sales_data.csv'
# df.to_csv(mac_linux_path, index=False)
mac_path='/Users/lingluli/Desktop/NTU DSAI Course/DSAI-M2-Assignment/Data/london_cycle_station_now.csv'
df.to_csv(mac_path)

(800, 16)
